# Galassi 2012 - Gate B: Solver Validation

This notebook implements Gate B of the validation pipeline. It loads the Gate A checkpoint, validates the solver against experimental data (if real), and handles placeholder data gracefully.

In [ ]:
import os
import sys
import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add repo to path
# Try to determine base path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/H2_Storage'
except ImportError:
    BASE_DIR = 'H2_Storage'
    print(f"Using local BASE_DIR: {BASE_DIR}")

REPO_DIR = os.path.join(BASE_DIR, 'repo')
sys.path.append(REPO_DIR)

try:
    from h2tank.galassi_baseline import simulate_fast_fill
except ImportError:
    # Attempt relative import if running as script from validation/notebooks
    sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../repo')))
    from h2tank.galassi_baseline import simulate_fast_fill

CHECKPOINTS_DIR = os.path.join(BASE_DIR, 'checkpoints')
FIGURES_DIR = os.path.join(BASE_DIR, 'figures')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')

# Ensure directories exist
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

## A) Load Gate A Checkpoint

In [ ]:
checkpoint_path = os.path.join(CHECKPOINTS_DIR, 'galassi2012_gateA.pkl')
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

with open(checkpoint_path, 'rb') as f:
    gate_a_data = pickle.load(f)

protocol_data = gate_a_data['protocol']
dataframes = gate_a_data['dataframes']
placeholder_flags = gate_a_data['placeholder_flags']

print("Loaded checkpoint.")
print("Protocol keys:", protocol_data.keys())
print("Placeholder flags:", placeholder_flags)

## Simulation Setup

In [ ]:
# Define model parameters
# These might need tuning, but we start with estimates or values from paper if known.
# Galassi 2012 mentions Type IV tank.
# For now, we use a baseline set of parameters.
# We will assume a small Volume ~40L (0.04 m3) typical for car tanks, or check if paper specified.
# Paper PDF not readable here, but prompt says "Replicate Table 1".
# If we don't know V, the temperature rise is independent of V for adiabatic, but dependent on UA/V for heat loss.
# We'll set a generic UA and V.
model_params = {
    'UA': 5.0,     # W/K, guess
    'vol_m3': 0.04 # m3, guess
}

results_sim = {}
metrics_sim = {}

## Run Simulations (Always Run)

In [ ]:
tests = ['H2_101', 'H2_250']

for test_name in tests:
    print(f"Running simulation for {test_name}...")
    protocol = protocol_data[test_name]
    
    # Run Solver
    res = simulate_fast_fill(protocol, model_params)
    results_sim[test_name] = res
    
    print(f"  Peak T: {res['T_peak_K']:.2f} K")

## B & C) Validation Logic (Real vs Placeholder)

In [ ]:
any_placeholder = any(placeholder_flags.values())
metrics_csv_path = os.path.join(RESULTS_DIR, 'galassi2012_gateB_metrics.csv')

if any_placeholder:
    print("\n!!! PLACEHOLDER DETECTED !!!")
    print("Marking results as NOT VALIDATED.")
    
    # Write INCOMPLETE metrics
    df_metrics = pd.DataFrame([{'Status': 'INCOMPLETE', 'Reason': 'Placeholder Data Detected'}])
    df_metrics.to_csv(metrics_csv_path, index=False)
    
    # Store incomplete status in metrics_sim for checkpoint
    metrics_sim['status'] = 'INCOMPLETE'
    metrics_sim['reason'] = 'Placeholder Data Detected'
    
    print(f"Wrote {metrics_csv_path}")
    print("STOP: Replace placeholder digitized CSVs.")
    
else:
    print("\nReal data detected. Proceeding with validation.")
    
    final_metrics_list = []
    pass_fail_checks = []
    
    for test_name in tests:
        sim = results_sim[test_name]
        # Get corresponding exp data
        # Map H2_101 -> fig5_TC5_H2_101_exp.csv
        filename = f"fig5_TC5_{test_name}_exp.csv"
        df_exp = dataframes[filename]
        
        # Metrics Calculation
        # 1. Peak Error %
        T_peak_sim = sim['T_peak_K']
        T_peak_exp = df_exp['T_K'].max()
        peak_error_pct = (T_peak_sim - T_peak_exp) / T_peak_exp * 100
        
        # 2. Time-to-peak Error
        # Find exp peak time
        idx_peak_exp = df_exp['T_K'].idxmax()
        t_peak_exp = df_exp.iloc[idx_peak_exp]['time_s']
        t_peak_sim = sim['t_peak_s']
        time_peak_error_s = t_peak_sim - t_peak_exp
        
        # 3. RMSE
        # Need to interpolate sim to exp time points
        T_sim_interp = np.interp(df_exp['time_s'], sim['time_s'], sim['T_K'])
        rmse = np.sqrt(np.mean((T_sim_interp - df_exp['T_K'])**2))
        
        # Peak rise for RMSE threshold
        peak_rise = T_peak_exp - df_exp['T_K'].min()
        rmse_threshold = 0.10 * peak_rise
        
        print(f"\nTest {test_name}:")
        print(f"  Peak T (Sim/Exp): {T_peak_sim:.1f} / {T_peak_exp:.1f} K (Err: {peak_error_pct:.2f}%)")
        print(f"  RMSE: {rmse:.2f} K (Threshold: {rmse_threshold:.2f} K)")
        
        # Pass/Fail Criteria for this test
        # PASS if peak error <= 10% AND RMSE <= 10% of observed peak rise
        pass_peak = abs(peak_error_pct) <= 10.0
        pass_rmse = rmse <= rmse_threshold
        test_passed = pass_peak and pass_rmse
        pass_fail_checks.append(test_passed)
        
        metrics_entry = {
            'Test': test_name,
            'Peak_Error_Pct': peak_error_pct,
            'RMSE_K': rmse,
            'Time_Peak_Error_s': time_peak_error_s,
            'Passed': test_passed
        }
        final_metrics_list.append(metrics_entry)
        
        # Plotting Overlays
        plt.figure(figsize=(10, 6))
        plt.plot(df_exp['time_s'], df_exp['T_K'], 'o', label='Exp')
        plt.plot(sim['time_s'], sim['T_K'], '-', label='Sim')
        plt.xlabel('Time (s)')
        plt.ylabel('Temperature (K)')
        plt.title(f'Galassi 2012 {test_name}: T vs t')
        plt.legend()
        plt.grid(True)
        fig_path = os.path.join(FIGURES_DIR, f'galassi2012_{test_name}_Tt_overlay.png')
        plt.savefig(fig_path)
        print(f"  Saved plot to {fig_path}")
        plt.close()
        
    # Save Metrics
    df_metrics = pd.DataFrame(final_metrics_list)
    df_metrics.to_csv(metrics_csv_path, index=False)
    print(f"\nSaved metrics to {metrics_csv_path}")
    
    # Store metrics in metrics_sim for checkpoint
    metrics_sim['data'] = final_metrics_list
    metrics_sim['status'] = 'COMPLETE'
    
    # D) Final Pass/Fail
    if all(pass_fail_checks):
        print("\nGATE B PASSED")
        metrics_sim['gate_status'] = 'PASSED'
    else:
        print("\nGATE B FAILED")
        metrics_sim['gate_status'] = 'FAILED'
    print(df_metrics)

## Save Checkpoint (Gate B)

In [ ]:
checkpoint_b_data = {
    'model_params': model_params,
    'results_sim': results_sim,
    'metrics': metrics_sim
}

checkpoint_b_path = os.path.join(CHECKPOINTS_DIR, 'galassi2012_gateB.pkl')
with open(checkpoint_b_path, 'wb') as f:
    pickle.dump(checkpoint_b_data, f)
print(f"Saved Gate B checkpoint to {checkpoint_b_path}")